# Transformatoren: magnetiske kretser, spoler og metning

## Pilotprosjekt for Matematikk 1

En transformator består i sin enkleste form av to viklinger som er koblet gjennom den samme magnetiske kjernen. Når strømmen og den magnetiske fluksen varierer, induseres spenninger i viklingene.

Studentene forventes ikke å ha hatt en full innføring i magnetisme. Derfor bygger prosjektet opp modellen trinnvis, med utgangspunkt i elektriske kretser og spolemodellen.

Prosjektet har fire hoveddeler:

1. **Magnetisk krets:** en analogi til elektriske motstandsnettverk
2. **Spole med jernkjerne:** sammenhengen mellom reluktans og induktans
3. **Lineær transformator:** et koblet $2\times2$-system for viklingsstrømmene
4. **Metning:** et tilstandsavhengig $3\times3$-system for strømmer og magnetisk fluks

Hysterese diskuteres som en mulig utvidelse, men inngår ikke i den obligatoriske modellen.

### Læringsmål

Etter prosjektet skal du kunne

- forklare de grunnleggende størrelsene i en magnetisk krets,
- bruke analogien mellom elektrisk motstand og magnetisk reluktans,
- sette opp og løse et lineært system for magnetisk fluks,
- utlede spolens induktans fra viklingstall og reluktans,
- løse en lineær førsteordens ODE analytisk og med Euler,
- implementere en tidsavhengig spenningskilde uten å bruke impedansregning,
- skrive en lineær transformator som et koblet ODE-system,
- undersøke inverterbarheten til en induktans- eller massematrise,
- løse et system med tilstandsavhengig massematrise ved hvert Euler-steg,
- forklare hvordan metning kan gi stor og forvrengt magnetiseringsstrøm.

### Modellnivå

Modellene er forenklede. Vi antar blant annet uniform fluks i hver kjernedel, konstante viklingsparametre og en glatt, entydig magnetiseringskurve i hovedmodellen. Full hysterese og virvelstrømstap krever mer avanserte materialmodeller.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Del A: Magnetiske kretser

## A.1 Kretsanalogien

En magnetisk krets er en forenklet modell av det magnetiske feltet i en jernkjerne. Vi samler feltet i en magnetisk fluks $\Phi$ gjennom kjernens tverrsnitt.

Vi bruker analogien

| Elektrisk krets | Magnetisk krets |
|---|---|
| spenning $V$ | magnetomotorisk kraft $F=Ni$ |
| strøm $I$ | magnetisk fluks $\Phi$ |
| motstand $R$ | reluktans $\mathcal R$ |
| $V=RI$ | $F=\mathcal R\Phi$ |

Reluktansen til en homogen kjernedel modelleres som

$$
\boxed{\mathcal R=\frac{\ell}{\mu A}},
$$

hvor

- $\ell$ er lengden av den magnetiske veien,
- $A$ er kjernens tverrsnittsareal,
- $\mu$ er materialets permeabilitet.

En lang og smal magnetisk vei har høy reluktans. En kort og tykk vei har lav reluktans.

**Begrensning ved analogien:** Magnetisk fluks er ikke en strøm av ladning eller stoff. Kretsanalogien er en matematisk modell av et feltproblem.

## Oppgave A1: Én magnetisk sløyfe

En vikling med $N$ vindinger fører strømmen $i$. Den magnetomotoriske kraften er

$$F=Ni.$$

For én lineær kjernedel gjelder

$$F=\mathcal R\Phi.$$

Bruk

$$
N=500,\quad i=0.20\ \mathrm A,
\quad \ell=0.40\ \mathrm m,
$$

$$
A=4.0\cdot10^{-4}\ \mathrm{m^2},
\quad \mu=2.0\cdot10^{-3}\ \mathrm{H/m}.
$$

Beregn $F$, $\mathcal R$, $\Phi$ og flukstettheten

$$B=\frac{\Phi}{A}.$$

In [ ]:
N = 500
I = 0.20
ell = 0.40
A_c = 4.0e-4
mu = 2.0e-3

F = ...
reluktans = ...
Phi = ...
B = ...

print("Magnetomotorisk kraft:", F, "A-vindinger")
print("Reluktans:", reluktans, "1/H")
print("Magnetisk fluks:", Phi, "Wb")
print("Flukstetthet:", B, "T")

## Oppgave A2: Kjernedeler i serie og et luftgap

Reluktanser i samme magnetiske fluksvei summeres:

$$
\mathcal R_{tot}=\mathcal R_1+\mathcal R_2+\cdots.
$$

En jernkjerne har et lite luftgap. Bruk

$$
\mathcal R_{jern}=\frac{\ell_{jern}}{\mu_{jern}A},
\qquad
\mathcal R_{gap}=\frac{\ell_{gap}}{\mu_0A}.
$$

Sammenlign bidragene dersom

$$
\ell_{jern}=0.40\ \mathrm m,\qquad
\ell_{gap}=1.0\ \mathrm{mm},
$$

$$
\mu_{jern}=2.0\cdot10^{-3}\ \mathrm{H/m},
\qquad
\mu_0=4\pi\cdot10^{-7}\ \mathrm{H/m}.
$$

Hvorfor kan et kort luftgap dominere totalreluktansen?

In [ ]:
mu0 = 4*np.pi*1e-7
ell_jern = 0.40
ell_gap = 1.0e-3
mu_jern = 2.0e-3

R_jern = ...
R_gap = ...
R_total = ...

print("Jernreluktans:", R_jern)
print("Gapreluktans:", R_gap)
print("Andel fra luftgap:", ...)

## A.2 En skallkjerne med delt fluks

Vi betrakter en forenklet magnetisk kjerne der fluksen deler seg i to returgrener:

```text
                         Phi_2
                 +----[ R_2 ]----+
                 |               |
F = N i ----[ R_1 ]              |
                 |               |
                 +----[ R_3 ]----+
                         Phi_3
```

Hovedfluksen er

$$\Phi_1=\Phi_2+\Phi_3.$$

De to magnetiske sløyfene gir

$$
F=\mathcal R_1\Phi_1+\mathcal R_2\Phi_2,
$$

$$
F=\mathcal R_1\Phi_1+\mathcal R_3\Phi_3.
$$

## Oppgave A3: Matriseform

Vis at fluksene oppfyller

$$
\begin{pmatrix}
1&-1&-1\\
\mathcal R_1&\mathcal R_2&0\\
\mathcal R_1&0&\mathcal R_3
\end{pmatrix}
\begin{pmatrix}
\Phi_1\\\Phi_2\\\Phi_3
\end{pmatrix}
=
\begin{pmatrix}
0\\F\\F
\end{pmatrix}.
$$

Løs systemet for en symmetrisk kjerne med $\mathcal R_2=\mathcal R_3$, og kontroller at $\Phi_2=\Phi_3$.

In [ ]:
R1 = 2.0e5
R2 = 5.0e5
R3 = 5.0e5
F = 100.0

A_mag = np.array([
    [1.0, -1.0, -1.0],
    [R1, R2, 0.0],
    [R1, 0.0, R3]
])

b_mag = np.array([0.0, F, F])

Phi_vec = ...
print("Phi1, Phi2, Phi3 =", Phi_vec)
print("Residual =", ...)

## Oppgave A4: Asymmetrisk kjerne

Øk $\mathcal R_3$ og undersøk hvordan fluksen fordeler seg. Anta at returgrenene har arealene $A_2$ og $A_3$. Beregn

$$B_2=\frac{\Phi_2}{A_2},\qquad
B_3=\frac{\Phi_3}{A_3}.
$$

Hvilken del kan først nå et gitt metningsnivå $B_{sat}$?

In [ ]:
R3_asym = 8.0e5
A2 = 2.0e-4
A3 = 1.5e-4
B_sat = 1.6

# Oppdater matrisen, løs og beregn B2 og B3.

# Del B: Fra spoleloven til en magnetisk kjerne

Studentene møter spolemodellen

$$
v_L=L\frac{di}{dt}
$$

i elektriske kretser. Nå viser vi hvordan induktansen kan knyttes til den magnetiske kretsen.

For en lineær kjerne gjelder

$$Ni=\mathcal R_m\Phi.$$

Flukskoblingen til en vikling med $N$ vindinger er

$$\lambda=N\Phi.$$

Dermed får vi

$$
\lambda
=N\frac{Ni}{\mathcal R_m}
=\frac{N^2}{\mathcal R_m}i.
$$

Vi definerer magnetiseringsinduktansen

$$
\boxed{L_m=\frac{N^2}{\mathcal R_m}}.
$$

Da er $\lambda=L_mi$. Spenningen over spolen er den tidsderiverte av flukskoblingen:

$$v_L=\frac{d\lambda}{dt}=L_m\frac{di}{dt}$$

når kjernen er lineær og $L_m$ er konstant.

## Oppgave B1: Beregn induktansen

Bruk reluktansen fra én magnetisk sløyfe og beregn

$$L_m=\frac{N^2}{\mathcal R_m}.$$

Undersøk hvordan induktansen påvirkes når

1. viklingstallet dobles,
2. tverrsnittet dobles,
3. kjernelengden dobles,
4. et luftgap innføres.

In [ ]:
N_B = 500
R_m_B = reluktans

L_m = ...
print("Magnetiseringsinduktans:", L_m, "H")

## B.2 En vikling med kobbermotstand

Dersom viklingen har motstanden $R$, gir Kirchhoffs spenningslov

$$
v(t)=Ri+L_m\dot i.
$$

Altså

$$
\boxed{
\dot i+\frac{R}{L_m}i=\frac{v(t)}{L_m}.
}
$$

Dette er en lineær førsteordens ODE.

## Oppgave B2: Konstant spenning

For $v(t)=V_0$ og $i(0)=i_0$ er løsningen

$$
i(t)=\frac{V_0}{R}+
\left(i_0-\frac{V_0}{R}\right)e^{-Rt/L_m}.
$$

Vis dette for hånd. Tidskonstanten er

$$\tau=\frac{L_m}{R}.$$

Bruk $V_0=5$ V, $R=10\ \Omega$ og $i_0=0$.

In [ ]:
V0 = 5.0
R_B = 10.0
i0 = 0.0

tau_B = ...
print("Tidskonstant:", tau_B, "s")

## Oppgave B3: Euler og eksakt løsning

Implementer Euler-metoden. Beregn også

$$
\Phi(t)=\frac{N}{\mathcal R_m}i(t),
\qquad
B(t)=\frac{\Phi(t)}{A_c}.
$$

Sammenlign Euler-løsningen med håndløsningen.

In [ ]:
def spole_konstant(t, i):
    di = ...
    return di


def euler_skalar(f, y0, T, h):
    n = int(round(T/h))
    t = np.linspace(0.0, n*h, n + 1)
    y = np.zeros(n + 1)
    y[0] = y0

    for k in range(n):
        y[k + 1] = ...

    return t, y


t_B, i_B = euler_skalar(spole_konstant, i0, T=5*tau_B, h=tau_B/100)
i_eksakt = ...
Phi_B = ...
B_B = ...

In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True)

ax[0].plot(t_B, i_B, label="Euler")
ax[0].plot(t_B, i_eksakt, "--", label="Eksakt")
ax[0].set_ylabel("Strøm A")
ax[0].legend()
ax[0].grid()

ax[1].plot(t_B, B_B)
ax[1].set_xlabel("Tid s")
ax[1].set_ylabel("Flukstetthet T")
ax[1].grid()

plt.show()

## Oppgave B4: Sinusformet spenning i tidsdomenet

Bruk den kjente inngangsfunksjonen

$$v(t)=V_{max}\sin(2\pi ft).$$

Studentene trenger ikke impedansregning. Euler-metoden evaluerer bare $v(t_n)$ i hvert tidssteg.

Bruk $f=10$ Hz og velg en steglengde som gir minst 200 steg per periode. Plott $v(t)$, $i(t)$ og $B(t)$.

In [ ]:
Vmax_B = 5.0
f_B = 10.0


def v_sinus(t):
    return Vmax_B*np.sin(2*np.pi*f_B*t)


def spole_sinus(t, i):
    return ...

periode_B = 1/f_B
h_B = periode_B/200

t_Bs, i_Bs = euler_skalar(spole_sinus, 0.0, T=5*periode_B, h=h_B)
v_Bs = v_sinus(t_Bs)
Phi_Bs = ...
B_Bs = ...

# Lag plott.

## Oppgave B5: Hvorfor er likespenning problematisk for en vanlig transformator?

I den forenklede modellen nærmer likestrømmen seg

$$i^*=\frac{V_0}{R}.$$

Når strømmen slutter å endre seg, blir den induktive spenningen liten. Strømmen begrenses da hovedsakelig av viklingsmotstanden.

Diskuter hvorfor en transformatorvikling som er dimensjonert for vekselspenning kan få svært stor strøm og oppvarming dersom den kobles til konstant likespenning.

# Del C: En lastet lineær transformator

## C.1 To viklinger, én felles kjerne

Vi bruker fortegnene slik at kjernens magnetomotoriske kraft er

$$F=N_1i_1-N_2i_2.$$

For en lineær kjerne med magnetisk konduktans

$$G_m=\frac{1}{\mathcal R_m}$$

blir fluksen

$$
\Phi=G_m(N_1i_1-N_2i_2).
$$

Transformatoren har også lekkasjeinduktansene $L_{\sigma1}$ og $L_{\sigma2}$. Disse representerer fluks som ikke kobler begge viklingene.

Med resistiv last $R_L$ får vi det koblede systemet

$$
\boxed{
\begin{pmatrix}
L_{\sigma1}+N_1^2G_m&-N_1N_2G_m\\
-N_1N_2G_m&L_{\sigma2}+N_2^2G_m
\end{pmatrix}
\begin{pmatrix}
\dot i_1\\\dot i_2
\end{pmatrix}
=
\begin{pmatrix}
v_1(t)-R_1i_1\\
-(R_2+R_L)i_2
\end{pmatrix}.
}
$$

## Oppgave C1: Inverterbarhet

Vis at determinantens kryssledd kansellerer, slik at

$$
\det L
=L_{\sigma1}L_{\sigma2}
+L_{\sigma1}N_2^2G_m
+L_{\sigma2}N_1^2G_m.
$$

Forklar hvorfor matrisen er inverterbar når

$$L_{\sigma1}>0,\qquad L_{\sigma2}>0,\qquad G_m>0.$$

Hvorfor er det bedre å bruke `np.linalg.solve` enn å beregne den inverse matrisen eksplisitt?

## Oppgave C2: Implementer lineær transformator

Bruk skalerte parameterverdier som gir moderate strømmer og tydelig kobling. Inngangsspenningen er sinusformet, men systemet løses direkte i tidsdomenet.

In [ ]:
N1 = 100
N2 = 50
R1 = 2.0
R2 = 0.50
R_L = 8.0
Lsig1 = 0.020
Lsig2 = 0.008
R_m = 2.0e6
G_m = 1/R_m

Vmax_C = 40.0
f_C = 20.0


def v1(t, fase=0.0):
    return Vmax_C*np.sin(2*np.pi*f_C*t + fase)


Lmat = np.array([
    [Lsig1 + N1**2*G_m, -N1*N2*G_m],
    [-N1*N2*G_m, Lsig2 + N2**2*G_m]
])

print("Induktansmatrise:
", Lmat)
print("Determinant:", ...)
print("Egenverdier:", ...)

In [ ]:
def lineær_transformator(t, x):
    i1, i2 = x
    rhs = np.array([
        v1(t) - R1*i1,
        -(R2 + R_L)*i2
    ])
    dx = ...
    return dx


def euler_system(f, x0, T, h):
    n = int(round(T/h))
    t = np.linspace(0.0, n*h, n + 1)
    X = np.zeros((n + 1, len(x0)))
    X[0] = x0

    for k in range(n):
        X[k + 1] = ...

    return t, X


periode_C = 1/f_C
h_C = periode_C/500

t_C, X_C = euler_system(
    lineær_transformator,
    x0=np.array([0.0, 0.0]),
    T=8*periode_C,
    h=h_C
)

i1_C = X_C[:, 0]
i2_C = X_C[:, 1]
Phi_C = ...
B_C = ...  # bruk et valgt kjerneareal

In [ ]:
A_core_C = 5.0e-4
B_C = Phi_C/A_core_C
v1_C = v1(t_C)
v_last_C = R_L*i2_C
P_last_C = R_L*i2_C**2

fig, ax = plt.subplots(3, 1, sharex=True, figsize=(8, 9))

ax[0].plot(t_C, v1_C, label="Primærspenning")
ax[0].plot(t_C, v_last_C, label="Lastspenning")
ax[0].set_ylabel("Spenning V")
ax[0].legend()
ax[0].grid()

ax[1].plot(t_C, i1_C, label="Primærstrøm")
ax[1].plot(t_C, i2_C, label="Sekundærstrøm")
ax[1].set_ylabel("Strøm A")
ax[1].legend()
ax[1].grid()

ax[2].plot(t_C, B_C)
ax[2].set_xlabel("Tid s")
ax[2].set_ylabel("Flukstetthet T")
ax[2].grid()

plt.show()

## Oppgave C3: Parameterstudier

Undersøk hvordan løsningen påvirkes når du varierer

1. lastmotstanden $R_L$,
2. vindingstallsforholdet $N_2/N_1$,
3. magnetisk reluktans $\mathcal R_m$,
4. lekkasjeinduktansene.

Sammenlign det beregnede spenningsforholdet med det ideelle forholdet

$$\frac{v_2}{v_1}\approx\frac{N_2}{N_1}.$$

Forklar hvorfor viklingsmotstand, lekkasjeinduktans og last gjør transformatoren mindre ideell.

# Del D: Transformator med metning

## D.1 En glatt magnetiseringskurve

I en lineær kjerne er $B=\mu H$. Ved metning blir sammenhengen ikke-lineær. Vi bruker den pedagogiske modellen

$$
\boxed{H(B)=aB+cB^3.}
$$

For liten $B$ dominerer det lineære leddet. Når $|B|$ øker, kreves en stadig større feltstyrke for å øke flukstettheten.

Den differensielle sammenhengen er

$$
\frac{dH}{dB}=a+3cB^2.
$$

Med magnetisk veilengde $\ell_c$ og kjerneareal $A_c$ blir den differensielle reluktansen

$$
\boxed{
\mathcal R_d(\Phi)
=\frac{\ell_c}{A_c}
\left[
a+3c\left(\frac{\Phi}{A_c}\right)^2
\right].
}
$$

## D.2 Tilstandsavhengig $3\times3$-system

Vi bruker tilstandsvektoren

$$x=(i_1,i_2,\Phi)^T.$$

Systemet er

$$
\boxed{
\begin{pmatrix}
L_{\sigma1}&0&N_1\\
0&L_{\sigma2}&-N_2\\
N_1&-N_2&-\mathcal R_d(\Phi)
\end{pmatrix}
\begin{pmatrix}
\dot i_1\\\dot i_2\\\dot\Phi
\end{pmatrix}
=
\begin{pmatrix}
v_1(t)-R_1i_1\\
-(R_2+R_L)i_2\\
0
\end{pmatrix}.
}
$$

Ved hvert Euler-steg beregnes $\mathcal R_d(\Phi_n)$, matrisen bygges, og den deriverte finnes med `np.linalg.solve`.

## Oppgave D1: Inverterbarhet

Determinanten er, opp til valgt fortegn,

$$
L_{\sigma1}L_{\sigma2}\mathcal R_d
+L_{\sigma1}N_2^2
+L_{\sigma2}N_1^2.
$$

Forklar hvorfor matrisen er inverterbar dersom

$$L_{\sigma1}>0,\quad L_{\sigma2}>0,
\quad \mathcal R_d(\Phi)>0.$$

Kontroller determinanten numerisk for et utvalg verdier av $\Phi$.

## Oppgave D2: Implementer metningsmodellen

Velg parameterne slik at modellen er omtrent lineær ved små flukstettheter, men får tydelig økende reluktans ved større $|B|$.

In [ ]:
ell_core = 0.50
A_core_D = 5.0e-4

a_mag = 400.0
c_mag = 250.0


def R_d(Phi):
    B = Phi/A_core_D
    return (ell_core/A_core_D)*(a_mag + 3*c_mag*B**2)


def mettet_transformator(t, x, fase=0.0):
    i1, i2, Phi = x

    M = np.array([
        [Lsig1, 0.0, N1],
        [0.0, Lsig2, -N2],
        [N1, -N2, -R_d(Phi)]
    ])

    rhs = np.array([
        v1(t, fase=fase) - R1*i1,
        -(R2 + R_L)*i2,
        0.0
    ])

    dx = ...
    return dx

In [ ]:
t_D, X_D = euler_system(
    lambda t, x: mettet_transformator(t, x, fase=0.0),
    x0=np.array([0.0, 0.0, 0.0]),
    T=8*periode_C,
    h=periode_C/1000
)

i1_D = X_D[:, 0]
i2_D = X_D[:, 1]
Phi_D = X_D[:, 2]
B_D = Phi_D/A_core_D

# Lag plott og sammenlign med den lineære modellen.

## Oppgave D3: Lineær kjerne mot mettende kjerne

Sammenlign

- toppverdien til primærstrømmen,
- formen på primærstrømmen,
- maksimal flukstetthet,
- sekundærstrøm og lastspenning.

Forklar hvorfor en kjerne som nærmer seg metning kan trekke stor magnetiseringsstrøm, selv om fluksen ikke øker tilsvarende mye.

## D.3 Innkoblingsstrøm

En transformator som kobles inn kan få en stor forbigående primærstrøm. Vi bruker

$$v_1(t)=V_{max}\sin(2\pi ft+\theta)$$

og undersøker hvordan responsen avhenger av

- innkoblingsvinkelen $\theta$,
- startfluksen $\Phi(0)$,
- spenningsamplituden,
- metningsparameteren $c$.

Startfluksen er en enkel representasjon av restfluks. Hovedmodellen beskriver ikke full hysterese.

## Oppgave D4: Innkoblingsvinkel og restfluks

Simuler minst disse tilfellene:

1. $\theta=0$ og $\Phi(0)=0$,
2. $\theta=\pi/2$ og $\Phi(0)=0$,
3. $\theta=0$ og positiv startfluks,
4. $\theta=0$ og negativ startfluks.

Sammenlign maksimal $|i_1|$ og maksimal $|B|$ de første periodene. Hvilken kombinasjon gir størst innkoblingsstrøm i modellen?

In [ ]:
tilfeller = [
    (0.0, 0.0, "theta=0, Phi0=0"),
    (np.pi/2, 0.0, "theta=pi/2, Phi0=0"),
    (0.0, 2.0e-4, "theta=0, positiv Phi0"),
    (0.0, -2.0e-4, "theta=0, negativ Phi0")
]

# Simuler hvert tilfelle, lagre toppverdier og lag sammenligningsplott.

# Del E: Hysterese som fordypning

## Hvorfor er hysterese vanskeligere?

I den mettende modellen er $H$ entydig bestemt av $B$:

$$H=H(B).$$

Ved hysterese kan samme feltstyrke gi forskjellige flukstettheter avhengig av tidligere magnetisering. Modellen trenger derfor hukommelse.

En avansert modell kan bruke interne tilstandsvariabler, for eksempel reversible og irreversible deler av magnetiseringen. Da lagres den relevante historien i tilstanden, ikke ved å gi ODE-en hele listen med tidligere punkter.

Referanseartikkelen bruker Jiles–Atherton- eller Tellinen-modeller for den inkrementelle permeabiliteten og inkluderer også virvelstrømstap. Dette er ikke obligatorisk i pilotprosjektet.

## En enkel pedagogisk minnemodell

En mulig demonstrasjonsmodell er

$$
\tau_M\dot M=M_s\tanh(H/H_0)-M,
$$

sammen med

$$B=\mu_0(H+M).$$

Denne gir en frekvensavhengig lukket kurve, men er ikke en fullverdig modell for rateuavhengig hysterese i transformatorstål. Den bør derfor merkes som en pedagogisk minnemodell.

# Modellkritikk

Diskuter minst fem av punktene:

- Magnetiske felt erstattes av samlede flukser.
- Flukstettheten antas uniform i hver kjernedel.
- Lekkasjefluksen representeres av konstante induktanser.
- Viklingsmotstandene er konstante og temperaturuavhengige.
- Metningskurven er en enkel polynommodell.
- Hysterese er ikke med i hovedmodellen.
- Virvelstrømstap er ikke med.
- Kapasitanser mellom viklinger er ikke med.
- Modellen er enfaset.
- Bryterøyeblikket modelleres uten kontaktfenomener.
- Eksplisitt Euler kan kreve svært liten steglengde.

## Mulig videreføring

- full hysterese med interne tilstander,
- virvelstrømstap,
- transformatorens temperatur,
- tre faser og flere kjernedeler,
- parameteridentifikasjon fra målinger,
- sammenligning med eksperimentell innkoblingsstrøm.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan magnetiske kretser kan sammenlignes med elektriske kretser,
2. hvordan reluktans bestemmes av lengde, areal og permeabilitet,
3. hvordan fluksbalansen ga et lineært system,
4. hvordan $L_m=N^2/\mathcal R_m$ ble utledet,
5. hvordan spole-ODE-en ble løst analytisk og med Euler,
6. hvordan to viklinger ble koblet gjennom samme kjerne,
7. hvorfor induktansmatrisen i den lineære modellen er inverterbar,
8. hvordan metning førte til en tilstandsavhengig massematrise,
9. hvordan innkoblingsvinkel og startfluks påvirket transienten,
10. hvorfor full hysterese krever en modell med hukommelse.

## Referanse for prosjektutviklingen

A. D. Theocharis, J. Milias-Argitis og Th. Zacharias, *Single-phase transformer model including magnetic hysteresis and eddy currents*, Electrical Engineering 90 (2008), 229–241.

Studentene trenger ikke lese forskningsartikkelen for å gjennomføre prosjektet.